# Qualitative Analysis — Base vs Fine-tuned Gemma 4 E2B

Side-by-side comparison of the base model and the LoRA-adapted artifact assessor.

**Run scripts 01–04 before opening this notebook.**

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'scripts'))
sys.path.insert(0, '../scripts')

import torch
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_from_disk
from transformers import AutoProcessor, Gemma4ForConditionalGeneration
from peft import PeftModel

from utils import DEVICE, DTYPE, MODEL_ID, ADAPTER_PATH, USER_PROMPT

print(f'Device: {DEVICE} | dtype: {DTYPE}')

## Load Both Models

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

# Base model (no adapter)
base_model = Gemma4ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map='auto' if DEVICE == 'cuda' else None,
)
if DEVICE == 'mps':
    base_model = base_model.to('mps')
base_model.eval()

# Fine-tuned model (with LoRA adapter)
ft_base = Gemma4ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map='auto' if DEVICE == 'cuda' else None,
)
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_PATH)
if DEVICE == 'mps':
    ft_model = ft_model.to('mps')
ft_model.eval()

print('Both models loaded.')

## Inference Helper

In [ ]:
@torch.no_grad()
def describe(model, image: Image.Image) -> str:
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text',  'text': USER_PROMPT},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt',
        tokenize=True,
    ).to(DEVICE)

    out_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    generated = out_ids[0][inputs['input_ids'].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True)


def compare(image: Image.Image, ground_truth: str = '') -> None:
    base_out = describe(base_model, image)
    ft_out   = describe(ft_model,   image)

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(image)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    if ground_truth:
        print(f'Ground truth :\n{ground_truth}\n')
    print(f'Base model   :\n{base_out}\n')
    print(f'Fine-tuned   :\n{ft_out}\n')
    print('─' * 60)

## Load Validation Set

In [ ]:
val_ds = load_from_disk('../data/vqa_val')
print(f'Validation samples: {len(val_ds):,}')

## Category 1 — Clear Anatomy Artifact

Pick an image with an obvious anatomy error (e.g. extra fingers).

In [ ]:
sample = val_ds[0]   # swap index or filter by label to find a good example
image  = sample['messages'][0]['content'][0]['image']
gt     = sample['messages'][1]['content']

compare(image, gt)

## Category 2 — Subtle Texture Artifact

In [ ]:
sample = val_ds[1]
image  = sample['messages'][0]['content'][0]['image']
gt     = sample['messages'][1]['content']

compare(image, gt)

## Category 3 — Clean Image (Hallucination Check)

The fine-tuned model should **not** invent artifacts here.

In [ ]:
sample = val_ds[2]
image  = sample['messages'][0]['content'][0]['image']
gt     = sample['messages'][1]['content']

compare(image, gt)

## Category 4 — Multi-Artifact Image

Tests whether the model enumerates multiple defects correctly.

In [ ]:
sample = val_ds[3]
image  = sample['messages'][0]['content'][0]['image']
gt     = sample['messages'][1]['content']

compare(image, gt)

## Custom Image

Drop any image path here to run inference outside the validation set.

In [ ]:
custom_image = Image.open('/path/to/your/image.jpg').convert('RGB')
compare(custom_image)